# SWOT File Sorter — Filter by Pass Number\n\nThis script:\n1. Reads a source folder containing cycle subfolders (e.g., `cycle_001`, `cycle_002`, etc.)\n2. Filters `.nc` files based on hardcoded pass numbers\n3. Copies matching files to a new output directory preserving the cycle folder structure

In [5]:
import os
import shutil
import re

# ============================================================
# CONFIGURATION — Edit these paths and pass numbers as needed
# ============================================================

# Source folder (can have any depth of subfolders — cycle folders, 
# forward/reproc groupings, etc.)
SOURCE_DIR = r"Y:\level 3 ocean basic files\Basic\reproc"

# Output folder where filtered files will be copied
OUTPUT_DIR = r"Y:\level 3 ocean basic files\Basic_filters"

# ---------- HARDCODED PASS NUMBERS YOU WANT TO KEEP ----------
DESIRED_PASSES = {
    "049", "077", "105", "133", "161",   # Group 1
    "355", "383", "411", "439",           # Group 2
    "174", "202", "230", "258",           # Group 3
    "452", "480", "508", "536",           # Group 4
}
# ============================================================


def extract_pass_number(filename):
    """
    Extract the 3-digit pass number from a SWOT filename.
    Pattern: ..._CCC_PPP_YYYYMMDD... where CCC=cycle, PPP=pass
    """
    match = re.search(r'_(\d{3})_(\d{3})_\d{8}T', filename)
    if match:
        return match.group(2)
    return None


def sort_and_copy_files(source_dir, output_dir, desired_passes):
    """
    Recursively walk through source_dir.
    For every .nc file whose pass number is in desired_passes,
    copy it to output_dir preserving the FULL relative folder structure.
    """
    total_copied = 0
    total_skipped = 0
    folder_summary = {}   # relative_folder → {copied, skipped, files}

    for root, dirs, files in os.walk(source_dir):
        # Get relative path from source root
        rel_path = os.path.relpath(root, source_dir)
        
        nc_files = [f for f in files if f.lower().endswith('.nc')]
        if not nc_files:
            continue
        
        copied_files = []
        skipped = 0
        
        for filename in sorted(nc_files):
            pass_num = extract_pass_number(filename)
            
            if pass_num and pass_num in desired_passes:
                # Build destination path preserving folder structure
                dest_folder = os.path.join(output_dir, rel_path)
                os.makedirs(dest_folder, exist_ok=True)
                
                src_path = os.path.join(root, filename)
                dst_path = os.path.join(dest_folder, filename)
                shutil.copy2(src_path, dst_path)
                copied_files.append(filename)
                total_copied += 1
            else:
                skipped += 1
                total_skipped += 1
        
        folder_summary[rel_path] = {
            'copied': len(copied_files),
            'skipped': skipped,
            'files': copied_files
        }

    # ---- Print Summary ----
    print("=" * 70)
    print(f"  SORTING COMPLETE")
    print(f"  Source : {source_dir}")
    print(f"  Output : {output_dir}")
    print(f"  Desired passes: {', '.join(sorted(desired_passes))}")
    print("=" * 70)
    print(f"\n  Total files copied  : {total_copied}")
    print(f"  Total files skipped : {total_skipped}")
    print(f"  Folders processed   : {len(folder_summary)}\n")
    
    for folder in sorted(folder_summary.keys()):
        info = folder_summary[folder]
        if info['copied'] > 0 or info['skipped'] > 0:
            print(f"  📁 {folder}/  ({info['copied']} copied, {info['skipped']} skipped)")
            for f in info['files']:
                print(f"      ✔ {f}")
    
    print("\n" + "=" * 70)
    return folder_summary


# ============================================================
# RUN
# ============================================================
print(f"Desired passes: {sorted(DESIRED_PASSES)}\n")
summary = sort_and_copy_files(SOURCE_DIR, OUTPUT_DIR, DESIRED_PASSES)

Desired passes: ['049', '077', '105', '133', '161', '174', '202', '230', '258', '355', '383', '411', '439', '452', '480', '508', '536']

  SORTING COMPLETE
  Source : Y:\level 3 ocean basic files\Basic\reproc
  Output : Y:\level 3 ocean basic files\Basic_filters
  Desired passes: 049, 077, 105, 133, 161, 174, 202, 230, 258, 355, 383, 411, 439, 452, 480, 508, 536

  Total files copied  : 739
  Total files skipped : 24532
  Folders processed   : 45

  📁 cycle_001/  (13 copied, 396 skipped)
      ✔ SWOT_L3_LR_SSH_Basic_001_161_20230726T224518_20230726T233644_v3.0.nc
      ✔ SWOT_L3_LR_SSH_Basic_001_174_20230727T095407_20230727T104533_v3.0.nc
      ✔ SWOT_L3_LR_SSH_Basic_001_202_20230728T095438_20230728T104604_v3.0.nc
      ✔ SWOT_L3_LR_SSH_Basic_001_230_20230729T095509_20230729T104635_v3.0.nc
      ✔ SWOT_L3_LR_SSH_Basic_001_258_20230730T095540_20230730T104707_v3.0.nc
      ✔ SWOT_L3_LR_SSH_Basic_001_355_20230802T210603_20230802T215729_v3.0.nc
      ✔ SWOT_L3_LR_SSH_Basic_001_383_20230803

In [6]:
# ============================================================
# VERIFY — Quick count of files in output directory
# ============================================================

def verify_output(output_dir):
    """Print a compact summary of the output directory."""
    if not os.path.exists(output_dir):
        print("Output directory does not exist yet. Run the sorting cell first.")
        return
    
    total = 0
    folder_count = 0
    for root, dirs, files in os.walk(output_dir):
        nc_files = [f for f in files if f.lower().endswith('.nc')]
        if nc_files:
            rel = os.path.relpath(root, output_dir)
            print(f"  📁 {rel}: {len(nc_files)} files")
            total += len(nc_files)
            folder_count += 1
    
    print(f"\n{'='*50}")
    print(f"  Total folders: {folder_count}")
    print(f"  Total .nc files: {total}")

verify_output(OUTPUT_DIR)

  📁 cycle_001: 13 files
  📁 cycle_002: 17 files
  📁 cycle_003: 17 files
  📁 cycle_004: 13 files
  📁 cycle_005: 17 files
  📁 cycle_006: 17 files
  📁 cycle_007: 17 files
  📁 cycle_008: 14 files
  📁 cycle_009: 17 files
  📁 cycle_010: 17 files
  📁 cycle_011: 17 files
  📁 cycle_012: 17 files
  📁 cycle_013: 17 files
  📁 cycle_014: 16 files
  📁 cycle_015: 13 files
  📁 cycle_016: 16 files
  📁 cycle_017: 17 files
  📁 cycle_018: 17 files
  📁 cycle_019: 17 files
  📁 cycle_020: 17 files
  📁 cycle_021: 17 files
  📁 cycle_022: 17 files
  📁 cycle_023: 16 files
  📁 cycle_024: 17 files
  📁 cycle_025: 17 files
  📁 cycle_026: 16 files
  📁 cycle_027: 17 files
  📁 cycle_028: 17 files
  📁 cycle_029: 17 files
  📁 cycle_030: 17 files
  📁 cycle_031: 16 files
  📁 cycle_032: 17 files
  📁 cycle_033: 17 files
  📁 cycle_034: 17 files
  📁 cycle_035: 17 files
  📁 cycle_036: 17 files
  📁 cycle_037: 17 files
  📁 cycle_038: 16 files
  📁 cycle_039: 17 files
  📁 cycle_040: 17 files
  📁 cycle_041: 17 files
  📁 cycle_042: 1